# Two-stage logical reconstruction (the schema-inference pipeline)

**Company server, PRIVATE data.** The approach `layout_description.md` calls for:
infer the logical schema, then assign every fragment to its cell — not conventional
TSR. Decoupled into two local stages:

1. **Stage 1** — OCR + geometry: PaddleOCR words → a grid/coords block, and
   (optionally) a MinerU2.5-Pro table-structure draft as the specialist ceiling.
2. **Stage 2** — a schema-conditioned reasoning VLM (thinking on) infers the logical
   schema from stage-1 evidence — **document-agnostic, no hard-coded layout** (a known
   header is passed only as an optional hint to verify) — emits logical HTML, keeps
   blanks blank, and tags each non-empty cell with a `data-bbox` so the cells can be drawn.

Input images are already **cropped to the table by a detector (YOLOX)**, so the prompt
also guards against a clipped outer border being read as a dropped/merged column.

## 1. Config

In [ ]:
import sys; sys.path.insert(0, "..")
from pathlib import Path

from src.model.registry import MODEL_QWEN36_27B
from src.model.vllm_client import VLLMTableReconstructor
from src.model.prompts import build_schema_instruction, DEFAULT_INVOICE_SCHEMA
from src.ocr.engine import run_ocr
from src.ocr.layout import serialize_layout

IMAGES_DIR = Path("data/invoices")
# Document-agnostic by default: infer the schema from the body, do NOT pin one layout
# (a prompt pinned to one sample only reconstructs documents that match it).
# with_bbox=True -> the model tags every non-empty cell with data-bbox so we can draw them.
# Pass a known header only as an optional hint to verify, e.g.:
#     build_schema_instruction(DEFAULT_INVOICE_SCHEMA, with_bbox=True)
INSTRUCTION = build_schema_instruction(with_bbox=True)
reasoner = VLLMTableReconstructor(model_id=MODEL_QWEN36_27B, base_url="http://localhost:8000/v1", thinking=True)
img = sorted(IMAGES_DIR.glob("*.png"))[0]
print("image:", img)

## 2. Stage 1a — OCR + geometry grounding

In [ ]:
layout = serialize_layout(run_ocr(img), style="grid")
print(layout[:1200])

## 3. Stage 1b (optional) — MinerU2.5-Pro structure draft (specialist ceiling)

A pure parser recovers the *visible* structure; it will miss implicit/optional
columns, which is exactly why stage 2 exists. Useful as a comparison floor.

In [ ]:
from src.model.mineru_client import MinerUTableReconstructor
mineru = MinerUTableReconstructor()
draft = mineru.predict(img)
print(draft.html[:1500] if draft.html else "(no table extracted — confirm MinerU output shape)")

## 4. Stage 2 — schema-conditioned reasoning → logical HTML

In [ ]:
# guided_regex forces output to a single <table>...</table> (data-bbox attrs allowed) and blocks stray prose.
pred = reasoner.predict(
    img, instruction=INSTRUCTION, ocr_layout=layout,
    guided_regex=r"<table>[\s\S]*</table>",
)
from IPython.display import HTML, Image as IPyImage, display
from src.demo.boxes import draw_cell_boxes, boxed_cells

display(HTML(pred.html or "<i>empty</i>"))

# Draw the per-cell boxes the model emitted (with_bbox) onto the cropped table:
# <th> header cells in blue, body cells in red; blank cells carry no box.
overlay = draw_cell_boxes(img, pred.html)
out = img.with_name(img.stem + "_boxes.png")
overlay.save(out)
print(f"drew {len(boxed_cells(pred.html))} cell boxes -> {out}")
display(overlay)

## 5. Compare the two stages

If you have any corrected label for this image, score both with the schema metrics
(`content_placement`, `blank_preservation`, `schema_match`) to see the reasoning
pass recover the implicit columns the specialist dropped.

In [ ]:
true_html = None   # paste a corrected label here to score
if true_html:
    from src.eval.metrics import content_placement, blank_preservation, schema_match
    for name, html in [("mineru", draft.html), ("stage-2", pred.html)]:
        cp, bp, sm = content_placement(html, true_html), blank_preservation(html, true_html), schema_match(html, true_html)
        print(f"{name:<8} placement={cp.accuracy:.2f} blank_keep={bp.rate:.2f} cols={sm.pred_cols}/{sm.true_cols} ok={sm.cols_correct}")

---
This is the reconstruction path the fine-tuned student is distilled to reproduce.